[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agustrionopj/python-mini-workshop/blob/main/ccd_preprocessing_astropy.ipynb?forcePlay=true)

# Pra-proses data CCD dengan `Astropy`

**Mini-workshop:** _Pelatihan Python — Imahnoong_  
**Pemateri:** Agus T. P. Jatmiko (Observatorium Bosscha)

**Tanggal:** 2026-10-31

---

<!-- > **TODO (before publishing):** replace `YOURUSER/YOURREPO` in the badge URL
> above and in the data-download cell (Section 1). There are exactly **2 spots**. -->

## ⚠️ Jangan edit file ini secara langsung!

Anda membuka file ini dari  **file asli** pada GitHub.
Gunakan dalam mode _read-only_:

- Anda bisa menjalankan seluruh _cell_ secara bebas dan mandiri.
- Jika ingin melakukan perubahan, lakukan hal berikut: `File → Save a copy in Drive` (simpan ke Drive anda sendiri).
- File asli pada repository tidak akan melihat perubahan yang anda lakukan.

Semua proses akan berlangsung di dalam Colab.

## 0. Learning objectives

After this workshop you should be able to:

1. Read FITS files and inspect their headers with `astropy.io.fits`
2. Build a **master bias**, **master dark**, and **master flat** by median combination
3. Understand why we median-combine instead of averaging single frames
4. Calibrate a raw science frame with the standard formula
5. Do the same workflow with `ccdproc`
6. Remove cosmic rays with `astroscrappy`

TODO: adapt these to your actual agenda.

In [ ]:
# --- Setup: install dependencies (Colab runtime is temporary, so run this each session) ---
!pip install -q astropy ccdproc astroscrappy

# TODO: pin versions if reproducibility matters for the workshop, e.g.:
# !pip install -q astropy==6.* ccdproc==2.* astroscrappy==1.*

In [ ]:
import glob
import os

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.stats import mad_std
import ccdproc
from astroscrappy import detect_cosmics

plt.rcParams['figure.dpi'] = 100

# Central place for paths used throughout the notebook
DATA_DIR = 'data'

os.makedirs(DATA_DIR, exist_ok=True)
print('Ready. Astropy version:', __import__('astropy').__version__)

## 1. Get the data

The calibration frames are stored in the workshop GitHub repository, grouped
by type. We fetch them straight into this Colab runtime.

> **TODO (before publishing):** set `DATA_BASE_URL` to your real repo,
> and adjust the file lists to the actual filenames you committed.

In [ ]:
# ---------------------------------------------------------------------------
# TODO: replace YOURUSER/YOURREPO with the real GitHub account/repo name
# ---------------------------------------------------------------------------
DATA_BASE_URL = "https://raw.githubusercontent.com/YOURUSER/YOURREPO/main/data"

# TODO: fill with your actual filenames, matching what you committed to the repo
BIAS_FILES = [
    "bias/bias_001.fits",
    "bias/bias_002.fits",
    # ...
]

DARK_FILES = [
    "dark/dark_001.fits",
    "dark/dark_002.fits",
    # ...
]

FLAT_FILES = [
    "flat/flat_001.fits",
    "flat/flat_002.fits",
    # ...
]

SCIENCE_FILES = [
    "science/science_001.fits",
    # ...
]

ALL_FILES = BIAS_FILES + DARK_FILES + FLAT_FILES + SCIENCE_FILES
print(f"Total files to fetch: {len(ALL_FILES)}")

In [ ]:
# Download the files (keeps the repo as the single source of truth)
for relpath in ALL_FILES:
    lastrun = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(lastrun), exist_ok=True)
    if not os.path.exists(lastrun):   # simple cache: skip if already fetched
        !wget -q -O {lastrun} {DATA_BASE_URL}/{relpath.replace(' ', '%20')}
        print("fetched:", relpath)

print("\nFiles on disk:")
!find {DATA_DIR} -name "*.fits" | sort

## 2. Inspect a raw frame

Before combining anything, always look at one frame and its header.
Key header keywords to check: `IMAGETYP` (frame type), `EXPTIME`,
`DATE-OBS`, `GAIN`, `SET-TEMP`.

**DISCUSS:** What should a pure bias frame look like? (No structure, no stars — just the readout pattern.)

In [ ]:
# Open the first science frame and look at its header
# TODO: pick a real filename from your list
SAMPLE = os.path.join(DATA_DIR, SCIENCE_FILES[0])

with fits.open(SAMPLE) as hdul:
    hdul.info()
    header = hdul[0].header.copy()
    data  = hdul[0].data.astype(float)

header  # last expression = nice HTML table in Colab

In [ ]:
def show_image(img, title="", ax=None):
    """Display with a percentile stretch (cheap equivalent of DS9's zscale)."""
    lo, hi = np.percentile(img, (5, 95))
    show = ax.imshow(img, cmap='gray', origin='lower', vmin=lo, vmax=hi)
    ax.set_title(title)
    return show

fig, ax = plt.subplots(figsize=(8, 8))
show_image(data, "RAW science frame", ax)
plt.show()

print(f"shape: {data.shape}")
print(f"basic stats:  min={data.min():.1f}  max={data.max():.1f}  "
      f"median={np.median(data):.1f}  MAD-std={mad_std(data):.1f}")

# TODO: guide participants — what do the hot columns, the glow in a corner,
# and the speckles (cosmic rays / stars) tell you about this detector?

## 3. Master Bias

Median-combining the bias frames rejects cosmic-ray hits and keeps the
fixed readout pattern (note: bias has no overscan trimming here —
TODO: add an overscan section if your camera has one).

**Why median?** A single cosmic ray in one outlier frame is discarded;
a mean would be dragged by it.

In [ ]:
def load_fits_list(file_list):
    """Load a list of FITS files into a stack of 2D float arrays."""
    frames = []
    for relpath in file_list:
        with fits.open(os.path.join(DATA_DIR, relpath)) as hdul:
            frames.append(hdul[0].data.astype(float))
    return np.array(frames)

# --- Master bias — median combine ----------------------------------------
bias_stack = load_fits_list(BIAS_FILES)              # TODO: confirm shapes match!
print("bias stack:", bias_stack.shape)

master_bias = np.median(bias_stack, axis=0)

fig, ax = plt.subplots(figsize=(6, 6))
show_image(master_bias, "MASTER BIAS", ax)
plt.show()

print(f"master bias: median={np.median(master_bias):.1f}  MAD-std={mad_std(master_bias):.1f}")

# TODO (exercise): compute master_bias with mean instead of median.
# Compare the MAD-std of both — which set of results do you prefer, and why?

## 4. Master Dark

Current accumulates roughly linearly with exposure time, so we combine a
set of darks of equal exposure and later **scale** by the science exposure.
For workshop simplicity we use the same exposure time as the science frames.

**DISCUSS:** How would you handle science frames of a different exposure?

In [ ]:

# --- Master dark — median combine, bias-subtracted -----------------------
dark_stack = load_fits_list(DARK_FILES)              # TODO: same-shape as bias?
print("dark stack:", dark_stack.shape)

master_dark_raw = np.median(dark_stack, axis=0)
master_dark     = master_dark_raw - master_bias      # count-rate per the dark exp time

# Exposure time to which master_dark corresponds (read from headers)
# TODO: adjust keyword to your camera, common names: EXPTIME, EXPOSURE, EXP_TIME
with fits.open(os.path.join(DATA_DIR, DARK_FILES[0])) as hdul:
    dark_exptime = float(hdul[0].header['EXPTIME'])
print(f"master dark corresponds to EXPTIME = {dark_exptime} s")

fig, ax = plt.subplots(figsize=(6, 6))
show_image(master_dark, "MASTER DARK (bias-subtracted)", ax)
plt.show()

# TODO (exercise): count pixels above some threshold (e.g. 50 e- in this time)
# — how many hot pixels does your detector have?

## 5. Master Flat

Combine flats, subtract bias, then normalize so the median is 1.
Dividing by this frame corrects pixel-to-pixel sensitivity, vignetting, dust.

In [ ]:
# --- Master flat — median combine, bias subtract, normalize ---------------
flat_stack = load_fits_list(FLAT_FILES)
print("flat stack:", flat_stack.shape)

master_flat_raw = np.median(flat_stack, axis=0) - master_bias
master_flat     = master_flat_raw / np.median(master_flat_raw)   # normalize to 1.0

fig, ax = plt.subplots(figsize=(6, 6))
show_image(master_flat, "MASTER FLAT (normalized)", ax)
plt.show()

print(f"flat signal level: median raw = {np.median(master_flat_raw):.1f} ADU")

# TODO (exercise): plot a horizontal cut across the flat center.
# Where is the vignetting strongest?

## 6. Calibrate the science frame

The standard reduction:

```
Calibrated = (Science - MasterBias - Dark_rate * t_exp) / MasterFlat
```

First with **plain numpy** so the arithmetic is visible, then (after)
the same thing with **ccdproc** as a built-in alternative.

In [ ]:
# --- Version A: manual numpy ---------------------------------------------
sci_data = data.copy()                               # loaded in Section 2
sci_exptime = float(header['EXPTIME'])               # TODO: adjust keyword

sci_debiased = sci_data - master_bias
sci_darkcorr = sci_debiased - master_dark * (sci_exptime / dark_exptime)

# Avoid dividing by ~0 in bad pixels
good = master_flat > 0.5                             # TODO: pick your threshold
sci_calibrated = np.where(good, sci_darkcorr / np.where(good, master_flat, 1.0), 0.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
show_image(np.clip(sci_data, np.percentile(sci_data, 5), np.percentile(sci_data, 95)),
           "RAW", axes[0])
show_image(sci_calibrated, "CALIBRATED (manual)", axes[1])
plt.suptitle("TODO: replace titles with object/filter/epoch")
plt.show()

print(f"backgrough level: raw={np.median(sci_data[slice(0,100),slice(0,100)]):.1f} "
      f"calibrated={np.median(sci_calibrated[slice(0,100),slice(0,100)]):.1f}")

# TODO (exercise): zoom in on a 200x200 cutout containing a star before and
# after calibration (e.g. with plt.imshow on a subregion). What changed?

In [ ]:
# --- Version B: the same pipeline with ccdproc ---------------------------
# ccdproc keeps the FITS header along with the pixels, and its units/hdr keywords are handy.

ccd_sci = ccdproc.CCDData.read(SAMPLE, unit='adu')  # TODO: unit depends on your camera

# subtract master bias (wrap the arrays into CCDData first)
ccd_bias = ccdproc.CCDData(master_bias, unit='adu')
ccd_dark = ccdproc.CCDData(master_dark, unit='adu')
ccd_flat = ccdproc.CCDData(master_flat, unit='dimensionless')

ccd_sci = ccdproc.subtract(ccd_sci, ccd_bias)
ccd_sci = ccdproc.subtract(ccd_sci, ccd_dark.multiply(
    ccdproc.CCDData(np.full_like(ccd_sci.data, sci_exptime / dark_exptime), unit='s')
))
ccd_sci = ccdproc.divide(ccd_sci, ccd_flat)

fig, ax = plt.subplots(figsize=(7, 7))
show_image(ccd_sci.data, "CALIBRATED (ccdproc)", ax)
plt.show()

# TODO (exercise): diff the manual and ccdproc results (sci_calibrated - ccd_sci.data).
# They should be ~identical modulo floating point. If not — can you find why?

## 7. Cosmic-ray removal

Calibration does not remove cosmic-ray events; for that use
`astroscrappy` (L.A.Cosmic algorithm, van Dokkum 2001).

In [ ]:
# --- Detect cosmic rays ---
crmask, cleaned = detect_cosmics(
    ccd_sci.data,
    sigclip=4.5,       # TODO: tune for your data quality
    invar=None,
    cleantype='medmask',
)
print(f"cosmic-ray pixels flagged: {crmask.sum()} "
      f"({100 * crmask.mean():.3f}% of the frame)")

fig, ax = plt.subplots(figsize=(7, 7))
show_image(crmask.astype(float), f"COSMIC-RAY MASK ({crmask.sum()} px)", ax)
plt.show()

# TODO (exercise): plot one flagged event zoomed in — a sharp triangle typically.
# TODO (exercise): change sigclip to 3.5 and 6, and see how the count changes.

## 8. Save the results

Write the calibrated frame back to a FITS file with a header that records
what was done (good practice — and needed if the file will be shot-processed later).

In [ ]:
# --- Save the calibrated frame ---
out_header = header.copy()
out_header['HISTORY'] = 'Calibrated in mini-workshop notebook'
out_header['HISTORY'] = 'master bias + scaled dark + flat correction'
out_header['HISTORY'] = 'cosmic rays removed with astroscrappy'

OUT_NAME = os.path.join(DATA_DIR, 'result_calibrated.fits')   # TODO: rename nicely

fits.PrimaryHDU(data=cleaned, header=out_header).writeto(OUT_NAME, overwrite=True)
print('written:', OUT_NAME)

# TODO (exercise): download it from Colab via the Files sidebar (left panel)
# or with:  from google.colab import files; files.download(OUT_NAME)

## 9. Exercises for participants (homework)

> Fill these in with tasks tailored to your class.

1. TODO — e.g. *rebuild the master flat for the other filter and compare the
   two flats pixel-by-pixel.*
2. TODO — e.g. *calibrate the science frame keeping the bad pixels masked,
   and visualise how the signal-to-noise changes.*
3. TODO — e.g. *measure the background standard deviation before and after
   flat fielding. By how much did the flat improve the noise?*

---

## References

- [`ccdproc` documentation](https://ccdproc.readthedocs.io/) — priorities and pitfalls
- [Astropy CCD data reduction notebook](https://learn.astropy.org/tutorials/)
- van Dokkum (2001), PASP 113, 1420 — L.A.Cosmic algorithm
